# Building a GPT from scratch (decoder-only Transformer)

I built the **full** Transformer from *Attention Is All You Need*:
an **encoder** (reads English) plus a **decoder** (writes Bangla).

A **GPT** is the same machine with the encoder thrown away. Only the decoder is left, and because
there is no encoder there is also no **cross-attention** block. What remains is very simple:

```
tokens -> embedding + positional encoding
       -> N x [ masked self-attention  ->  feed forward ]
       -> projection to vocabulary
```

Its only job is: **given the words so far, predict the next word.** Train that on your own text
and the model starts writing in the style of that text.

We will go step by step:
1. Configuration
2. Load your custom text data
3. Build the tokenizer (turn words into numbers)
4. Prepare the data for the model
5. Build the model (decoder-only)
6. Train the model
7. Generate text


In [1]:
import math
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Split
from tokenizers.decoders import Fuse
from tokenizers import Regex

## Step 1: Configuration

All the settings for our training run live in one place: the `config` dictionary.

Our dataset is a plain text file (`tiny-shakespeare.txt`) containing whatever text we want the model
to learn to imitate - own writing, song lyrics, stories, chat logs, anything.

 Custom datasets are usually small, and a huge model on a small dataset just memorizes it.

In [2]:
config = {
    # data
    "dataset_path": "docs/tiny-shakespeare.txt",

    # tokenizer
    "tokenizer_dir": "tokenizers",

    # model size
    "seq_len": 128,         # how many characters the model can look back at
    "d_model": 256,

    "num_layers": 4,        # number of decoder blocks (GPT has only decoder blocks)

    "num_heads": 4,

    "d_ff": 1024,

    "dropout": 0.1,

    # training

    "batch_size": 32,
    "num_epochs": 20,

    "lr": 3e-4,

    # where to save trained model checkpoints

    "model_dir": "weights",
    "model_basename": "gpt_model_",
}

## Step 2: Load custom text data

We just read the whole text file into one big string.

> Tip: if  data is a CSV, read it with pandas and join one column into a single string, e.g.
> `text = "\n".join(df["my_column"].dropna())`, then write it to `docs/mydata.txt`.

In [3]:
SAMPLE_TEXT = """
the sun rose over the quiet city and the streets began to fill with people .
a small boy walked to school with his bag on his back .
the river moved slowly under the old bridge .
birds sang in the trees near the water .
an old man sat on a bench and read his newspaper .
the city was loud but the morning felt calm .
cars passed by and the shops opened one after another .
a woman sold flowers at the corner of the street .
children laughed as they ran across the park .
the day was bright and the sky was clear .
"""


def load_text(config):
    path = Path(config["dataset_path"])

    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(SAMPLE_TEXT.strip(), encoding="utf-8")
        print(f"No dataset found, wrote a tiny sample dataset to {path}")
        print("Replace this file with your own text and re-run!")

    text = path.read_text(encoding="utf-8")

    print(f"Loaded {len(text)} characters from {path}")
    print(f"Roughly {len(text.split())} words")

    return text


text = load_text(config)
print("\nFirst 300 characters:\n")
print(text[:300])

Loaded 1115394 characters from docs\tiny-shakespeare.txt
Roughly 202651 words

First 300 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Step 3: Build the tokenizer

Neural networks only understand numbers, not words. A **tokenizer** turns text into a list of
numbers.

** we split on every single character instead of
every word.** So `"the cat"` becomes `['t','h','e',' ','c','a','t']`, not `['the','cat']`.

Why bother? A word-level vocabulary on a corpus like Shakespeare is huge - tens of thousands of
distinct words, most of which appear only once or twice. The model barely gets to see most of them,
so it can't learn to generalize and just memorizes exact phrases instead (train loss goes down,
validation loss doesn't move). A character-level vocabulary is tiny - around 60-90 symbols for
English text - and every single one of them shows up thousands of times. That is what lets the
model actually learn *patterns* instead of memorizing.

The trade-off: the model now has to work harder, since it must learn to spell words one letter at
a time before it can learn grammar and structure on top of that. That's why we also gave it a
longer lookback window (`seq_len`) above - a character carries far less information than a whole
word, so the model needs to see more of them at once to have enough context.

Same special tokens as before:
- `[UNK]` - a character the tokenizer has never seen
- `[PAD]` - padding (barely used here, kept for consistency)
- `[SOS]` - start of text
- `[EOS]` - end of text

In [18]:
def get_or_build_tokenizer(config, text):

    tokenizer_path = Path(config["tokenizer_dir"]) / "gpt_tokenizer.json"

    tokenizer_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if tokenizer_path.exists():

        tokenizer = Tokenizer.from_file(str(tokenizer_path))

        print(f"Loaded tokenizer from {tokenizer_path}")

    else:

        print("Building tokenizer...")

        tokenizer = Tokenizer(
            WordLevel(unk_token="[UNK]")
        )

        trainer = WordLevelTrainer(
            special_tokens=[
                "[UNK]",  # unknown character
                "[PAD]",  # padding
                "[SOS]",  # start-of-text marker
                "[EOS]"   # end-of-text marker
            ],
            min_frequency=1
        )

        #  split into individual characters instead of whitespace-separated words
        tokenizer.pre_tokenizer = Split(pattern=Regex("."), behavior="isolated")

        # and un-split them the same way on the way out, with no extra spaces inserted
        tokenizer.decoder = Fuse()

        # train_from_iterator wants an iterable of strings, so we hand it the lines of our text
        tokenizer.train_from_iterator(
            text.splitlines(),
            trainer
        )

        tokenizer.save(str(tokenizer_path))

        print(f"Saved tokenizer to {tokenizer_path}")

    return tokenizer


tokenizer = get_or_build_tokenizer(config, text)

print(f"Vocabulary size: {tokenizer.get_vocab_size()} different characters")
print("Example:", tokenizer.encode("the sun rose over the city").ids)

Loaded tokenizer from tokenizers\gpt_tokenizer.json
Vocabulary size: 13359 different characters
Example: [8, 435, 2260, 619, 8, 506]


## Step 4: Turn the text into training examples

Here we have one long stream of words, and each example is just a
window of that stream:

```
full text ids:  [ 12  45  9  78  3  61  22  7 ... ]

input  (x):     [ 12  45  9  78 ]
label  (y):     [ 45   9 78   3 ]      <- the same window shifted one step to the left
```

So at every position the model sees the words so far and must predict the **very next** word.
Position 0 predicts position 1, position 1 predicts position 2, and so on - the whole window is
trained at once.

We also need the **causal mask**, d
number 3 the model may only look at words 1 and 2, never ahead.

In [5]:
def causal_mask(size):
    # Returns a (1, size, size) mask that is 1 on and below the diagonal, 0 above it.
    # "1" means "allowed to look at this position", "0" means "not allowed" (future word).
    mask=torch.tril(torch.ones((1, size, size))).int()
    return mask

In [6]:
class GPTDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        super().__init__()
        self.token_ids = token_ids      # one long python list of ids for the whole text
        self.seq_len = seq_len

    def __len__(self):
        # every starting position that still has seq_len + 1 tokens after it
        return max(0, len(self.token_ids) - self.seq_len)

    def __getitem__(self, idx):
        # a window of seq_len + 1 tokens
        chunk = self.token_ids[idx : idx + self.seq_len + 1]

        # input  = first seq_len tokens
        decoder_input = torch.tensor(chunk[:-1], dtype=torch.int64)   # (seq_len,)

        # label  = the same window shifted by one (what should come next at each position)
        label = torch.tensor(chunk[1:], dtype=torch.int64)            # (seq_len,)

        # security check
        assert decoder_input.size(0) == self.seq_len
        assert label.size(0) == self.seq_len

        # hide future words (no [PAD] handling needed - every window is completely full)
        decoder_mask = causal_mask(self.seq_len)                      # (1, seq_len, seq_len)

        return {
            "decoder_input": decoder_input,   # (seq_len)
            "decoder_mask": decoder_mask,     # (1, seq_len, seq_len)
            "label": label                    # (seq_len)
        }

## Step 5: Create the DataLoaders

We encode the whole text once, split the token stream 90% / 10% into train and validation, and
wrap both in `DataLoader`s so batching and shuffling are done for us.

The validation part is never trained on - we only use it to check the model is actually learning
the language rather than memorizing the exact text.

In [7]:
def get_dataloaders(config, text, tokenizer):

    # encode the ENTIRE text into one long list of token ids
    token_ids = tokenizer.encode(text).ids

    print(f"Total tokens in dataset: {len(token_ids)}")

    if len(token_ids) <= config["seq_len"] + 1:
        raise ValueError("Dataset is too small for this seq_len - add more text or lower seq_len")

    # 90% train, 10% validation (we split the stream, not random windows,
    # so validation text is truly unseen)
    train_size = int(0.9 * len(token_ids))

    train_ids = token_ids[:train_size]
    val_ids = token_ids[train_size:]

    train_ds = GPTDataset(train_ids, config["seq_len"])
    val_ds = GPTDataset(val_ids, config["seq_len"])

    print(f"Training windows: {len(train_ds)}, validation windows: {len(val_ds)}")

    train_dataloader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)

    if len(val_ds) == 0:
        # the 10% validation slice is shorter than one window - happens with tiny datasets
        print("Not enough text for a validation set, skipping validation. Add more text!")
        val_dataloader = None
    else:
        val_dataloader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

    return train_dataloader, val_dataloader

## Step 6: Build the model


### 6.1 Input embeddings and positional encoding

In [8]:
class InputEmbeddings(nn.Module):
    """Turns a token id into a d_model-sized vector."""

    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # (batch, seq_len) -> (batch, seq_len, d_model)
        # the sqrt(d_model) scaling is from the paper for normalization
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    """Adds information about WHERE each word sits in the sentence."""

    def __init__(self, d_model, max_seq_length, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # a (max_seq_length, d_model) matrix of sine/cosine waves
        pe = torch.zeros(max_seq_length, d_model)

        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)  # (max_seq_length, 1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)   # even positions
        pe[:, 1::2] = torch.cos(position * div_term)   # odd positions

        pe = pe.unsqueeze(0)  # (1, max_seq_length, d_model)

        # register_buffer = "save this with the model, but it is not a trainable weight"
        self.register_buffer("pe", pe)

    def forward(self, x):
        # add the positional wave for each position of the current sequence
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

### 6.2 Layer norm, feed forward, and the residual connection

In [9]:
class LayerNormalization(nn.Module):
    """Keeps the numbers in each vector in a healthy range so training stays stable."""

    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features))   # multiplied
        self.bias = nn.Parameter(torch.zeros(features))   # added

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias


class FeedForwardBlock(nn.Module):
    """A small 2-layer network applied to every position separately."""

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (batch, seq_len, d_model) -> (batch, seq_len, d_ff) -> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))


class ResidualConnection(nn.Module):
    """x + sublayer(norm(x)) - lets the original signal flow around each block."""

    def __init__(self, features, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization(features)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

### 6.3 Multi-head self-attention



In [10]:
class MultiHeadAttentionBlock(nn.Module):

    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_k = d_model // num_heads   # size of each head

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout):
        d_k = query.shape[-1]

        # (batch, num_heads, seq_len, d_k) @ (batch, num_heads, d_k, seq_len)
        #                       -> (batch, num_heads, seq_len, seq_len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            # wherever the mask is 0, make the score -infinity so softmax gives it 0 weight
            attention_scores.masked_fill_(mask == 0, -1e9)

        attention_scores = attention_scores.softmax(dim=-1)

        if dropout is not None:
            attention_scores = dropout(attention_scores)

        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        # split d_model into num_heads pieces:
        # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.num_heads, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.num_heads, self.d_k).transpose(1, 2)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # glue the heads back together: (batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.num_heads * self.d_k)

        return self.w_o(x)

### 6.4 The decoder block



A GPT block has only **two** - 

In [11]:
class DecoderBlock(nn.Module):

    def __init__(self, features, self_attention_block, feed_forward_block, dropout):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block

        # only 2 residual connections now (the translation decoder had 3)
        self.residual_connections = nn.ModuleList([
            ResidualConnection(features, dropout) for _ in range(2)
        ])

    def forward(self, x, mask):
        # 1) masked self-attention: every word looks at the words before it
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, mask))

        # 2) feed forward
        x = self.residual_connections[1](x, self.feed_forward_block)

        return x


class Decoder(nn.Module):
    """A stack of DecoderBlocks, followed by a final layer norm."""

    def __init__(self, features, layers):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class ProjectionLayer(nn.Module):
    """Turns each d_model vector into one score per word in the vocabulary."""

    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # (batch, seq_len, d_model) -> (batch, seq_len, vocab_size)
        return self.proj(x)

### 6.5 Putting it together: the GPT model



In [12]:
class GPT(nn.Module):

    def __init__(self, decoder, embed, pos, projection_layer):
        super().__init__()
        self.decoder = decoder
        self.embed = embed
        self.pos = pos
        self.projection_layer = projection_layer

    def decode(self, x, mask):
        # (batch, seq_len) -> (batch, seq_len, d_model)
        x = self.embed(x)
        x = self.pos(x)
        return self.decoder(x, mask)

    def projection_to_vocab(self, x):
        # (batch, seq_len, d_model) -> (batch, seq_len, vocab_size)
        return self.projection_layer(x)

    def forward(self, x, mask):
        x = self.decode(x, mask)
        return self.projection_to_vocab(x)


class BuildGPT:
    """Assembles all the pieces above into one model - """

    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_seq_length, dropout):
        self.num_layers = num_layers
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.dropout = dropout

    def get_model(self):
        # embeddings + positions
        embed = InputEmbeddings(self.d_model, self.vocab_size)
        pos = PositionalEncoding(self.d_model, self.max_seq_length, self.dropout)

        # the stack of decoder blocks
        decoder_blocks = []
        for _ in range(self.num_layers):
            self_attention_block = MultiHeadAttentionBlock(self.d_model, self.num_heads, self.dropout)
            feed_forward_block = FeedForwardBlock(self.d_model, self.d_ff, self.dropout)
            decoder_blocks.append(
                DecoderBlock(self.d_model, self_attention_block, feed_forward_block, self.dropout)
            )

        decoder = Decoder(self.d_model, nn.ModuleList(decoder_blocks))

        projection_layer = ProjectionLayer(self.d_model, self.vocab_size)

        model = GPT(decoder, embed, pos, projection_layer)

        # a good starting point for the weights makes training much easier
        for p in model.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        return model


def build_model(config, vocab_size):
    builder = BuildGPT(
        num_layers=config["num_layers"],
        d_model=config["d_model"],
        num_heads=config["num_heads"],
        d_ff=config["d_ff"],
        vocab_size=vocab_size,
        max_seq_length=config["seq_len"],
        dropout=config["dropout"]
    )
    return builder.get_model()

## Step 7: Train the model

For every batch we:
1. Run the words through the model to get a prediction for the next word at every position
2. Compare those predictions to the `label` (the same window shifted by one) using a loss function
3. Adjust the model's weights a tiny bit to reduce that loss (`backward()` + `step()`)

After each epoch we also run over the validation windows **without** updating the weights, so we
can see whether the model is still improving on text it has never trained on.

A useful number to watch is **perplexity** = `exp(loss)`. Very roughly: "how many words the model
is choosing between at each step". Lower is better.

In [13]:
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            decoder_input = batch["decoder_input"].to(device)
            decoder_mask = batch["decoder_mask"].to(device)
            label = batch["label"].to(device)

            output = model(decoder_input, decoder_mask)

            loss = loss_fn(output.view(-1, output.size(-1)), label.view(-1))
            total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)


def train_model(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    Path(config["model_dir"]).mkdir(parents=True, exist_ok=True)

    text = load_text(config)
    tokenizer = get_or_build_tokenizer(config, text)

    train_dataloader, val_dataloader = get_dataloaders(config, text, tokenizer)

    model = build_model(config, tokenizer.get_vocab_size()).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model has {num_params:,} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], eps=1e-9)

    pad_token_id = tokenizer.token_to_id("[PAD]")
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id, label_smoothing=0.1).to(device)

    for epoch in range(config["num_epochs"]):
        model.train()
        total_loss = 0.0

        for batch in train_dataloader:
            decoder_input = batch["decoder_input"].to(device)   # (batch, seq_len)
            decoder_mask = batch["decoder_mask"].to(device)     # (batch, 1, seq_len, seq_len)
            label = batch["label"].to(device)                   # (batch, seq_len)

            # forward pass: predict the next word at every position
            output = model(decoder_input, decoder_mask)
            # output: (batch, seq_len, vocab_size)

            # compare predictions to the shifted window
            loss = loss_fn(output.view(-1, output.size(-1)), label.view(-1))

            # update the model's weights
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_dataloader)

        if val_dataloader is not None:
            val_loss = evaluate(model, val_dataloader, loss_fn, device)
            print(f"Epoch {epoch + 1}/{config['num_epochs']} - "
                  f"train loss: {avg_loss:.4f} - val loss: {val_loss:.4f} - "
                  f"val perplexity: {math.exp(min(val_loss, 20)):.1f}")
        else:
            print(f"Epoch {epoch + 1}/{config['num_epochs']} - train loss: {avg_loss:.4f}")

    # save the trained model so we don't have to retrain it every time
    final_path = Path(config["model_dir"]) / f"{config['model_basename']}final.pt"
    torch.save({"model_state_dict": model.state_dict()}, final_path)
    print(f"Saved trained model to {final_path}")

    return model, tokenizer, device

## Step 8: Generate text

`greedy_decode()` 
output to feed in, so we only keep appending to the sequence.

1. Turn the prompt into token ids
2. Ask the model for the next word
3. Append it and repeat
4. Stop at `[EOS]` or when we hit `max_new_tokens`

Two ways to pick the next word:

- **Greedy** (`temperature=0`): always take the single most likely word. Safe, but repetitive -
  it often gets stuck in loops like "the the the".
- **Sampling** (`temperature > 0`): draw a word at random, weighted by the probabilities.
  `temperature` controls the risk-taking: 0.7 is fairly sensible, 1.2 is wild.
  `top_k` first throws away everything except the `k` most likely words, which stops the model
  from picking something absurd.

One extra detail: our positional encoding only knows `seq_len` positions, so if the generated text
gets longer than that we feed the model only the **last `seq_len` tokens** (a sliding window).

In [14]:
@torch.no_grad()
def generate(model, tokenizer, prompt, config, device, max_new_tokens=50, temperature=0.8, top_k=20):
    model.eval()

    eos_id = tokenizer.token_to_id("[EOS]")
    seq_len = config["seq_len"]

    # turn the prompt into token ids: (1, prompt_len)
    token_ids = tokenizer.encode(prompt).ids
    if len(token_ids) == 0:
        token_ids = [tokenizer.token_to_id("[SOS]")]

    decoder_input = torch.tensor([token_ids], dtype=torch.int64).to(device)

    for _ in range(max_new_tokens):

        # the model can only look at seq_len tokens, so keep the most recent ones
        window = decoder_input[:, -seq_len:]

        mask = causal_mask(window.size(1)).to(device)

        out = model.decode(window, mask)

        # we only care about the prediction made at the LAST position
        logits = model.projection_to_vocab(out[:, -1])   # (1, vocab_size)

        if temperature <= 0:
            # greedy: take the most likely word
            next_token = torch.argmax(logits, dim=-1)
        else:
            logits = logits / temperature

            if top_k is not None:
                # keep only the top_k scores, set the rest to -infinity
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = -float("inf")

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).squeeze(1)

        # append the new word and continue
        decoder_input = torch.cat([decoder_input, next_token.unsqueeze(0)], dim=1)

        if next_token.item() == eos_id:
            break

    return tokenizer.decode(decoder_input.squeeze(0).detach().cpu().numpy())

## Step 9: Run it all

This trains the GPT on your text and then writes some new text.

In [15]:
model, tokenizer, device = train_model(config)

Using device: cuda
Loaded 1115394 characters from docs\tiny-shakespeare.txt
Roughly 202651 words
Loaded tokenizer from tokenizers\gpt_tokenizer.json
Total tokens in dataset: 261973
Training windows: 235647, validation windows: 26070
Model has 10,008,623 parameters
Epoch 1/20 - train loss: 3.3260 - val loss: 8.4482 - val perplexity: 4666.7
Epoch 2/20 - train loss: 1.9122 - val loss: 8.4404 - val perplexity: 4630.2
Epoch 3/20 - train loss: 1.7625 - val loss: 8.3086 - val perplexity: 4058.6
Epoch 4/20 - train loss: 1.6986 - val loss: 8.2781 - val perplexity: 3936.6
Epoch 5/20 - train loss: 1.6612 - val loss: 8.2568 - val perplexity: 3853.8
Epoch 6/20 - train loss: 1.6356 - val loss: 8.2261 - val perplexity: 3737.2
Epoch 7/20 - train loss: 1.6170 - val loss: 8.1854 - val perplexity: 3588.3
Epoch 8/20 - train loss: 1.6029 - val loss: 8.1840 - val perplexity: 3583.0
Epoch 9/20 - train loss: 1.5917 - val loss: 8.1379 - val perplexity: 3421.9
Epoch 10/20 - train loss: 1.5826 - val loss: 8.1812

In [16]:
generate(model, tokenizer, "KING RICHARD", config, device,
         max_new_tokens=50, temperature=0.8, top_k=20)


'KING RICHARD III : Then he must die to - day ; For now he hath two deep bosom my soul brought forth ; The rest of that consorted crew , Destruction straight shall dog them at the heels . Good uncle , help to order several powers To Oxford , or'

### Try different prompts and settings

Greedy vs. sampling makes a big difference - run these a few times and compare.

In [17]:
print("GREEDY   :", generate(model, tokenizer, "the city", config, device, max_new_tokens=30, temperature=0))
print()
print("SAMPLING :", generate(model, tokenizer, "the city", config, device, max_new_tokens=30, temperature=0.8, top_k=20))
print()
print("WILD     :", generate(model, tokenizer, "the city", config, device, max_new_tokens=30, temperature=1.3, top_k=50))

GREEDY   : the city ; and all the instruments which aided to expose the child were even then lost when it was found . But O , the noble combat that ' twixt joy

SAMPLING : the city ' s institutions , and the terms For common justice , you ' re as pregnant in As art and practise hath enriched any That we remember . There is

WILD     : the city : To rouse the same ancient thoughts from mayst thou live Once more we of thee ; haply look sweetly apothecary ,-- You kiss thy guilty of my . Alack


## What changed vs.  Transformer - a summary

| | Translation Transformer | GPT (this notebook) |
|---|---|---|
| Architecture | Encoder + Decoder | Decoder only |
| Tokenizers | 2 (English, Bangla) | 1 |
| Attention per block | self + cross + feed forward | self + feed forward |
| Training example | sentence pair | a window of text and the same window shifted by 1 |
| Masks | padding mask + causal mask | causal mask only |
| Task | translate a sentence | predict the next word |
| At inference | encode once, decode word by word | just decode word by word |

## Things to try next

1. **Use real data.** Drop a bigger `docs/mydata.txt` in - the sample file is far too small to
   produce anything sensible. Aim for at least a few hundred KB.
2. **Switch to a BPE tokenizer.** A word-level tokenizer gives `[UNK]` for every word it did not
   see in training. Swapping `WordLevel` for `BPE` (from `tokenizers.models`) fixes that and
   shrinks the vocabulary a lot.
3. **Grow the model** once the data is bigger: `d_model=512`, `num_layers=6`, `seq_len=128`.
4. **Tie the weights.** Real GPTs share the embedding matrix with the projection layer:
   `model.projection_layer.proj.weight = model.embed.embedding.weight`. Fewer parameters, better results.
5. **Gradient clipping** (`torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` before
   `optimizer.step()`) keeps training stable on larger datasets.
6. **Learning-rate warmup**, exactly as in the paper - helps a lot once the model gets bigger.